# Faz 3 — Sentetik Veri Üretimi ve Kalite Testleri

**Sorumlu:** Ümit (Veri Mimarı ve Kalite Uzmanı)

Bu notebook proje şartnamesindeki Faz 3 gereksinimlerini uygular:
1. **İki Farklı Sentetik Veri Tekniği:** `imbalanced-learn` (`SMOTE`) ve `sdv` (`CTGANSynthesizer`) ile yalnızca eğitim seti (`X_train`) üzerinden sentetik örnek üretimi.
2. **İstatistiksel Kalite Ölçümü:** Gerçek ve sentetik dağılımlar arasında **Kolmogorov-Smirnov (KS) Testi**, **Wasserstein Mesafesi** ve **`|Corr_Real - Corr_Synthetic|`** korelasyon farkı analizi.
3. **Makine Öğrenmesi Fayda Testleri (`TSTR vs. TRTR`):** Saf sentetik eğitim (`TSTR`), gerçek eğitim (`TRTR`) ve `1:1` dengelenmiş eğitim senaryolarının dokunulmamış gerçek test seti (`X_test`) üzerinde **Macro F1, Weighted F1, ROC-AUC ve PR-AUC** ile karşılaştırılması (`Accuracy` kullanılmaz).

In [1]:
import time

import pandas as pd

from src import config
from src import synthetic as syn

## 1. Faz 2'den İşlenmiş Eğitim ve Test Verilerinin Yüklenmesi

In [2]:
X_train = pd.read_pickle(config.PROCESSED_DATA_DIR / 'X_train.pkl')
X_test = pd.read_pickle(config.PROCESSED_DATA_DIR / 'X_test.pkl')
y_train = pd.read_pickle(config.PROCESSED_DATA_DIR / 'y_train.pkl')
y_test = pd.read_pickle(config.PROCESSED_DATA_DIR / 'y_test.pkl')
df_missing = pd.read_pickle(
    config.PROCESSED_DATA_DIR / 'df_missing.pkl'
)

print(f'X_train : {X_train.shape} | Sınıf 0: '
      f'{int((y_train == 0).sum()):,} | Sınıf 1: '
      f'{int((y_train == 1).sum()):,}')
print(f'X_test  : {X_test.shape}  | Sınıf 0: '
      f'{int((y_test == 0).sum()):,} | Sınıf 1: '
      f'{int((y_test == 1).sum()):,}')

X_train : (81412, 154) | Sınıf 0: 72,326 | Sınıf 1: 9,086
X_test  : (20354, 154)  | Sınıf 0: 18,083 | Sınıf 1: 2,271


## 2. Teknik 1 (`SMOTE`) ve Teknik 2 (`CTGAN`) ile Sentetik Üretim

Veri sızıntısını önlemek için her iki üretici de **yalnızca `X_train`** üzerinde eğitilir. Hem azınlık sınıfını `1:1` oranında dengeleyen veri setleri hem de hiçbir gerçek hasta satırı içermeyen `%100` saf sentetik `TSTR` veri setleri üretilir.

In [3]:
t0 = time.perf_counter()
smote_data = syn.generate_smote_datasets(X_train, y_train)
smote_sec = time.perf_counter() - t0

print(f'SMOTE süre               : {smote_sec:.2f} sn')
print(f'SMOTE 1:1 dengeli boyut  : '
      f"{smote_data['X_balanced'].shape}")
print(f'SMOTE saf TSTR boyut     : '
      f"{smote_data['X_tstr'].shape}")

SMOTE süre               : 2.22 sn
SMOTE 1:1 dengeli boyut  : (144652, 154)
SMOTE saf TSTR boyut     : (18000, 154)


In [4]:
t0 = time.perf_counter()
ctgan_data = syn.generate_ctgan_datasets(
    X_train, y_train, df_missing, fit_rows_per_class=6000, epochs=50
)
ctgan_sec = time.perf_counter() - t0

print(f'CTGAN süre               : {ctgan_sec:.2f} sn')
print(f'CTGAN 1:1 dengeli boyut  : '
      f"{ctgan_data['X_balanced'].shape}")
print(f'CTGAN saf TSTR boyut     : '
      f"{ctgan_data['X_tstr'].shape}")

CTGAN süre               : 291.11 sn
CTGAN 1:1 dengeli boyut  : (144652, 154)
CTGAN saf TSTR boyut     : (18000, 154)


## 3. İstatistiksel Kalite Ölçümü (KS Testi, Wasserstein ve `|Corr_Real - Corr_Synthetic|`)

In [5]:
X_real_pos = X_train[y_train == 1]
fid_smote = syn.evaluate_statistical_fidelity(
    X_real_pos, smote_data['X_syn_pos']
)
fid_ctgan = syn.evaluate_statistical_fidelity(
    X_real_pos, ctgan_data['X_syn_pos']
)

fidelity_comp = pd.DataFrame({
    'Özellik': fid_smote['Özellik'],
    'SMOTE_KS': fid_smote['KS_İstatistiği'],
    'CTGAN_KS': fid_ctgan['KS_İstatistiği'],
    'SMOTE_1-KS': fid_smote['KS_Benzerlik_(1-KS)'],
    'CTGAN_1-KS': fid_ctgan['KS_Benzerlik_(1-KS)'],
    'SMOTE_Wasserstein': fid_smote['Wasserstein_Mesafesi'],
    'CTGAN_Wasserstein': fid_ctgan['Wasserstein_Mesafesi'],
})
fidelity_comp.round(4)

,Özellik,SMOTE_KS,CTGAN_KS,SMOTE_1-KS,CTGAN_1-KS,SMOTE_Wasserstein,CTGAN_Wasserstein
0,time_in_hospital,0.0885,0.1335,0.9115,0.8665,0.0891,0.1649
1,num_lab_procedures,0.0362,0.2170,0.9638,0.7830,0.0767,0.4130
2,num_procedures,0.1434,0.3712,0.8566,0.6288,0.0819,0.1743
3,num_medications,0.0464,0.1496,0.9536,0.8504,0.0656,0.2813
4,number_outpatient,0.1358,0.4631,0.8642,0.5369,0.0642,0.1589
5,number_emergency,0.1038,0.5357,0.8962,0.4643,0.0562,0.1663
6,number_inpatient,0.1530,0.3280,0.8470,0.6720,0.0874,0.3736
7,number_diagnoses,0.1403,0.3198,0.8597,0.6802,0.0862,0.1157
8,diag_1,0.0661,0.1144,0.9339,0.8856,0.0922,0.1686
9,diag_2,0.0697,0.2730,0.9303,0.7270,0.1052,0.4059


In [6]:
corr_rows = [
    {'Yöntem': 'SMOTE (Sentetik Azınlık)',
     **syn.correlation_difference_summary(
         X_real_pos, smote_data['X_syn_pos']
     )},
    {'Yöntem': 'CTGAN (Sentetik Azınlık)',
     **syn.correlation_difference_summary(
         X_real_pos, ctgan_data['X_syn_pos']
     )},
    {'Yöntem': 'SMOTE (Saf TSTR Genel)',
     **syn.correlation_difference_summary(
         X_train, smote_data['X_tstr']
     )},
    {'Yöntem': 'CTGAN (Saf TSTR Genel)',
     **syn.correlation_difference_summary(
         X_train, ctgan_data['X_tstr']
     )},
]
pd.DataFrame(corr_rows).round(4)

,Yöntem,Ortalama_|ΔCorr|,Medyan_|ΔCorr|,Maksimum_|ΔCorr|
0,SMOTE (Sentetik Azınlık),0.0104,0.0065,0.0566
1,CTGAN (Sentetik Azınlık),0.0880,0.0497,0.5240
2,SMOTE (Saf TSTR Genel),0.0170,0.0132,0.0717
3,CTGAN (Saf TSTR Genel),0.0999,0.0724,0.5175


## 4. Makine Öğrenmesi Fayda Testleri (`TSTR vs. TRTR`)

Tüm senaryolar **yalnızca gerçek `X_test` (`20.354` hasta)** üzerinde değerlendirilir. Şartname gereği `Accuracy` yerine **`Macro_F1`, `Weighted_F1`, `ROC_AUC` ve `PR_AUC`** raporlanır.

In [7]:
scenarios = {
    'TRTR (Gerçek Eğitim)': (X_train, y_train),
    'TSTR — SMOTE (%100 Sentetik)': (
        smote_data['X_tstr'], smote_data['y_tstr']
    ),
    'TSTR — CTGAN (%100 Sentetik)': (
        ctgan_data['X_tstr'], ctgan_data['y_tstr']
    ),
    'Real + SMOTE (1:1 Dengeli)': (
        smote_data['X_balanced'], smote_data['y_balanced']
    ),
    'Real + CTGAN (1:1 Dengeli)': (
        ctgan_data['X_balanced'], ctgan_data['y_balanced']
    ),
}

utility_df = syn.evaluate_ml_utility(scenarios, X_test, y_test)
utility_df.round(4)

,Senaryo,Model,Eğitim_Satır,Pozitif_Oran,Macro_F1,Weighted_F1,ROC_AUC,PR_AUC
0,TRTR (Gerçek Eğitim),LogisticRegression,81412,0.1116,0.5227,0.7185,0.6739,0.2140
1,TRTR (Gerçek Eğitim),HistGradientBoosting,81412,0.1116,0.5208,0.7102,0.6722,0.2217
2,TSTR — SMOTE (%100 Sentetik),LogisticRegression,18000,0.5000,0.5107,0.7014,0.6639,0.2069
3,TSTR — SMOTE (%100 Sentetik),HistGradientBoosting,18000,0.5000,0.4470,0.6029,0.6455,0.2018
4,TSTR — CTGAN (%100 Sentetik),LogisticRegression,18000,0.5000,0.4936,0.6786,0.6531,0.1881
5,TSTR — CTGAN (%100 Sentetik),HistGradientBoosting,18000,0.5000,0.5087,0.7119,0.6306,0.1655
6,Real + SMOTE (1:1 Dengeli),LogisticRegression,144652,0.5000,0.5163,0.7100,0.6658,0.2090
7,Real + SMOTE (1:1 Dengeli),HistGradientBoosting,144652,0.5000,0.5133,0.8440,0.6419,0.1964
8,Real + CTGAN (1:1 Dengeli),LogisticRegression,144652,0.5000,0.5292,0.7548,0.6226,0.1680
9,Real + CTGAN (1:1 Dengeli),HistGradientBoosting,144652,0.5000,0.4959,0.8398,0.6636,0.2029


## 5. Sentetik Veri Setlerinin Kaydedilmesi (Faz 4 & Faz 5 İçin)

In [8]:
smote_data['X_syn_pos'].to_pickle(
    config.PROCESSED_DATA_DIR / 'X_syn_smote_pos.pkl'
)
ctgan_data['X_syn_pos'].to_pickle(
    config.PROCESSED_DATA_DIR / 'X_syn_ctgan_pos.pkl'
)
smote_data['X_tstr'].to_pickle(
    config.PROCESSED_DATA_DIR / 'X_tstr_smote.pkl'
)
smote_data['y_tstr'].to_pickle(
    config.PROCESSED_DATA_DIR / 'y_tstr_smote.pkl'
)
ctgan_data['X_tstr'].to_pickle(
    config.PROCESSED_DATA_DIR / 'X_tstr_ctgan.pkl'
)
ctgan_data['y_tstr'].to_pickle(
    config.PROCESSED_DATA_DIR / 'y_tstr_ctgan.pkl'
)
print('Kaydedilen sentetik dosyalar:', [
    p.name for p in sorted(
        config.PROCESSED_DATA_DIR.glob('*syn*.pkl')
    ) + sorted(config.PROCESSED_DATA_DIR.glob('*tstr*.pkl'))
])

Kaydedilen sentetik dosyalar: ['X_syn_ctgan_pos.pkl', 'X_syn_smote_pos.pkl', 'X_tstr_ctgan.pkl', 'X_tstr_smote.pkl', 'y_tstr_ctgan.pkl', 'y_tstr_smote.pkl']
